In [1]:
import sys
from pathlib import Path

parent_dir = str(Path.cwd().parent)
sys.path.append(parent_dir)

import torch
import numpy as np
import matplotlib.pyplot as plt
from pydicom import dcmread
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

from pydose_rt.MachineConfig import MachineConfig
from pydose_rt.DoseEngine import DoseEngine

/home/rd/anaconda3/envs/autoplan/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-09-30 13:59:03.925480: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE4.1 SSE4.2 AVX AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
config = MachineConfig(ct_array_shape=(167, 167, 185), resolution=(0.3, 0.3, 0.3), field_size=(10, 10), number_of_leaf_pairs=5, tpr_20_10=0.72, number_of_cps=1, mu_scaling=1.8, starting_angle=0, iso_center=(15.35, 0.0, 0.0))
print(config)

ct_array_shape=(167, 167, 185) resolution=(0.3, 0.3, 0.3) field_size=(10, 10) downsampling_factor=(1, 1, 1) iso_center=(15.35, 0.0, 0.0) minimum_leaf_overlap=0.5 maximum_leaf_speed=2.25 minimum_gantry_angle_speed=0.1 maximum_gantry_angle_speed=6.0 maximum_gantry_angle_speed_variation=0.75 minimum_dose_rate=50.0 maximum_dose_rate=600.0 is_fff=True mu_scaling=1.8 focal_spot_sigma=0.15 focus_to_collimator=49.7 oar_coeffs=(1.0, -0.0001, 2.5e-07) mlc_thickness=6.8 mlc_mu=0.7 dtype=torch.float32 device=device(type='cuda') number_of_leaf_pairs=5 tpr_20_10=0.72 mean_photon_energy_MeV=10.0 SID=100.0 number_of_cps=1 starting_angle=0.0 mlc_transmission=0.00856560939749806 gantry_angles=array([0.]) depth_offset=84.65 gantry_diff=6.283185307179586 gantry_diff_deg=360.0 iso_center_in_pixels=array([52,  0,  0], dtype=int32) field_size_in_pixels=(34, 34) leaf_size=2.0 leaf_widths=array([2., 2., 2., 2., 2.], dtype=float32) fluence_profile=(array([ 0.  ,  1.  ,  2.  ,  3.  ,  4.  ,  5.  ,  7.5 , 10.  , 

In [3]:
x_ct = 0.0 * np.expand_dims(np.ones(config.ct_array_shape), 0)
# x_ct[:, 50:120, 50:120, 65:120] = 0
# x_ct[:, 80:90, 80:90, 75:110] = 1000
y_mlc = np.zeros((1, 2, config.number_of_cps, config.number_of_leaf_pairs))
y_mlc[:, 0, :, :] = 0.5
y_mlc[:, 1, :, :] = 1.0
mus = np.ones((1, config.number_of_cps), dtype=np.float32)
dose_layer = DoseEngine(config, 55)
dose = dose_layer(torch.tensor(x_ct, dtype=torch.float32, device=device), torch.tensor(y_mlc, dtype=torch.float32, device=device), torch.tensor(mus, dtype=torch.float32, device=device)).detach().cpu().numpy()

ValueError: CT image must be provided either at initialization or in forward().

In [ ]:
slice_idx = 84
print(dose.shape)

# plt.imshow(x_ct[0, :, :, 80], cmap='gray', vmin=-1, vmax=1)
plt.imshow(dose[0, :, 85, :], cmap='jet', vmin=np.min(dose), vmax=np.max(dose)) #, alpha=0.5)
plt.axis('off')
plt.colorbar()
# plt.axis('off')
plt.show()

In [ ]:
path ="/home/bolo/Downloads/10x10-10MV/RD1.2.752.243.1.1.20240927183310596.8800.73001.dcm"
path_ct = "/home/bolo/Downloads/10x10-10MV/CT1.2.752.243.1.1.20210112110104939.5560.87015.dcm"
ds = dcmread(path)
ds_ref = ds.pixel_array * float(ds.DoseGridScaling)
ds_ref = np.transpose(ds_ref, (1, 2, 0))[1:-1, 1:-1, :]
print(ds_ref.shape)

plt.imshow(ds_ref[:, 84, :], cmap='jet')
plt.axis('off')
plt.colorbar()
plt.show()

In [ ]:
slice_idx = 84
print(dose.shape)
print(ds_ref.shape)

# dose_plot = dose / dose[0, 34, 84, 92] * ds_ref[34, 84, 92]
dose_plot = dose / np.max(dose) * np.max(ds_ref)
# dose = dose * np.max(ds_ref) / np.max(dose)
plt.subplot(131)
plt.imshow(dose_plot[0, :, :, 92], cmap='jet')
plt.axis('off')
plt.subplot(132)
plt.imshow(ds_ref[:, :, 92], cmap='jet')
plt.axis('off')
plt.subplot(133)
plt.imshow((ds_ref[:, :, 92] - dose_plot[0, :, :, 92]) / (ds_ref[:, :, 92] + 1e-10), cmap='coolwarm', vmin=-1, vmax=1)
plt.colorbar()
plt.axis('off')
plt.show()

In [ ]:
air_gap = 1

print(config)

slice_idx = 84
depth_index = 93

x = np.linspace(0, config.ct_array_shape[0], config.ct_array_shape[0])

# Create the plot
plt.plot(x, dose_plot[0, :, slice_idx, depth_index], label='TensorFlow', color='orange')
plt.plot(x, ds_ref[:, slice_idx, depth_index], label='Phantom', color='blue')
plt.legend()

# Set x-ticks from 0 to 128, but label them as 0 to 26
num_ticks = 10
new_xticks = np.linspace(0, config.ct_array_shape[0], num_ticks)  # 27 points from 0 to 128
new_xtick_labels = np.round(np.linspace(0, config.ct_array_shape[0]*config.resolution[0], num_ticks), 1)  # Corresponding labels from 0 to 26

# Apply the new ticks and labels
plt.xticks(new_xticks, new_xtick_labels)
plt.xlabel('[cm]')

# Show the plot
plt.show()

In [ ]:
print(config)

x = np.linspace(0, config.ct_array_shape[0], config.ct_array_shape[0])

# Create the plot
slices = np.linspace(10, config.ct_array_shape[0], 3, endpoint=False).astype(int)
line_styles = ['-', '--', ':']
for i in range(len(slices)):
  plt.plot(x, ds_ref[slices[i], :, slice_idx], label=f'Phantom (d={int(slices[i] * config.resolution[1])} cm)', color='blue', linestyle=line_styles[i])
  plt.plot(x, dose_plot[0, slices[i], :, depth_index], label=f'PyTorch (d={int(slices[i] * config.resolution[1])} cm)', color='orange', linestyle=line_styles[i])
plt.legend()

# Apply the new ticks and labels
plt.xticks(new_xticks, new_xtick_labels)
plt.xlabel('[cm]')

# Show the plot
plt.show()